# Introduction

**In this notebook, I'm going to train Google T5 model to translate from English to Roman Urdu. The T5 model was presented in Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer by Colin Raffel, Noam Shazeer, Adam Roberts, Katherine Lee, Sharan Narang, Michael Matena, Yanqi Zhou, Wei Li, Peter J. Liu. **

This notebook is partly based on the tutorial provided on Hugging Face Transformer at the following link https://huggingface.co/docs/transformers/tasks/translation

# Imports

In [1]:
!pip install -q --upgrade rouge-score
!pip install --upgrade keras tensorflow keras_nlp
!pip install sacrebleu evaluate



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.1/644.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 51.4 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 23.5.26
    Uninstalling flatbuffers-23.5.26:
      Successfully uninstalled flatbuffers-23.5.26
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.2.0
    Uninstalling ml-dtypes-0.2.0:
      Successfully uninstalled ml-dtypes-0.2.0
  Attempting uninstall: keras
    Found existing installation: keras 3.2.1
    Uninstalling keras-3.2.1:
      Successfully uninstalled keras-3.2.1
  Attempting uninstall: h5py
    Found existing installation: h5py 3.10.0
    Uninstalling h5py-3.10.0:
      Successf

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import random
from transformers import AutoTokenizer, DataCollatorForSeq2Seq, AdamWeightDecay, TFAutoModelForSeq2SeqLM
from datasets import Dataset
from collections import Counter
from difflib import SequenceMatcher
import tensorflow as tf
import sacrebleu
from transformers.optimization_tf import AdamWeightDecay



# Optional: Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print(f"TensorFlow version: {tf.__version__}")

print("\nGPU Available: ", tf.config.list_physical_devices('GPU'))

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

2024-11-16 18:38:32.988537: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-16 18:38:32.988595: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-16 18:38:32.990135: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow version: 2.15.1

GPU Available:  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
/kaggle/input/weights-t5/t5_translation_weights.h5
/kaggle/input/erupd-nmt/ERUPD_NMT.csv


# Preprocess Dataframe

In [3]:
# Load and split the data
df = pd.read_csv('/kaggle/input/erupd-nmt/ERUPD_NMT.csv', encoding='latin')
df['English'] = df['English'].astype(str)
df['Roman Urdu'] = df['Roman Urdu'].astype(str)


# Train, validation, and test split
train_df = df.sample(frac=0.8, random_state=42)
temp_df = df.drop(train_df.index).reset_index(drop=True)
valid_df = temp_df.sample(frac=0.75, random_state=42)
test_df = temp_df.drop(valid_df.index).reset_index(drop=True)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

In [4]:
df.shape

(75146, 2)

In [5]:
df.head()

,English,Roman Urdu
0,"In the heart of the bustling city, lived a you...",Shehar ki dil mein rehti thi ek nojawan aurat ...
1,She had always dreamed of exploring the world ...,Usne hamesha khwab dekha tha ke apne sheher ke...
2,"One day, as she perused a travel magazine, Ais...","Ek din, jab usne ek safar nama parha, Aisha ne..."
3,The vivid descriptions of lush landscapes and ...,Yehan ke hari bhari manazir aur gaon walon ki ...
4,"Determined to turn her dreams into reality, Ai...",Apne khwabon ko haqeeqat mein tabdeel karne ka...


There are like 75,146 records in the dataframe.

Define a create text pairs fuction where each text pair contains English and its Roman Urdu version.

In [6]:
def create_text_pairs(dataframe):
    return list(zip(dataframe['English'], dataframe['Roman Urdu']))

# Preprocess Model

Load tokenizer from pre-trained model Goolgle T5. Here we just use the small model.

In [7]:
# Initialize tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('google-t5/t5-small')
model = TFAutoModelForSeq2SeqLM.from_pretrained('google-t5/t5-small')
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, return_tensors="tf")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFT5ForConditionalGeneration.

All the weights of TFT5ForConditionalGeneration were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFT5ForConditionalGeneration for predictions without further training.


In [8]:
optimizer = AdamWeightDecay(learning_rate=1e-4, weight_decay_rate=0.01)
prefix = 'Translate English to Roman Urdu: '

Load a pre-trained Google T5-small model under TFAutoModelForSeq2SeqLM.

In order to train and fine-tune t5 model, we need to have a prefix for a task since the model can do multi-task. Each task has a unique prefix. For translation, the prefix is: 'translate x to y' where x and y stand for name of language.

# Prepare Dataset

Create a function preprocess dataset to prepare the tensorflow dataset for model inputs.

In [9]:
def preprocess_dataset(text_pairs):
    en_texts, ur_texts = zip(*text_pairs)
    inputs = [prefix + text for text in en_texts]
    targets = ur_texts
    tokenized_data = tokenizer(inputs, text_target=targets, max_length=64, truncation=True)
    tf_dataset = model.prepare_tf_dataset(Dataset.from_dict(tokenized_data),
                                          shuffle=True, batch_size=64,
                                          collate_fn=data_collator)
    
    return tf_dataset


In [10]:
train_set = preprocess_dataset(create_text_pairs(train_df))
validation_set = preprocess_dataset(create_text_pairs(valid_df))


# Fine-tune Model

In [11]:
# model.compile(optimizer=optimizer)


In [12]:
# # Train the model and save weights
# model.fit(x=train_set, validation_data=validation_set, epochs=60)
# model.save_weights('t5_translation_weights.h5')


In [13]:
# model.save_pretrained("/kaggle/working/saved_model/")
# tokenizer.save_pretrained("/kaggle/working/tokenizer/")


# Test Model

In [14]:
model.load_weights("/kaggle/input/weights-t5/t5_translation_weights.h5")

In [15]:
def custom_meteor_score(reference, hypothesis):
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()
    
    # Calculate precision, recall, and F-score
    ref_counts = Counter(ref_tokens)
    hyp_counts = Counter(hyp_tokens)
    matches = sum(min(ref_counts[word], hyp_counts[word]) for word in hyp_counts)
    precision = matches / len(hyp_tokens) if hyp_tokens else 0
    recall = matches / len(ref_tokens) if ref_tokens else 0
    f_score = (10 * precision * recall) / (9 * precision + recall) if precision + recall > 0 else 0
    
    # Calculate fragmentation penalty based on sequence gaps
    matcher = SequenceMatcher(None, ref_tokens, hyp_tokens)
    match_blocks = matcher.get_matching_blocks()
    fragmentation = sum(1 for i in range(len(match_blocks) - 1) if match_blocks[i].size > 0)
    penalty = 0.5 * (fragmentation / len(hyp_tokens)) if hyp_tokens else 1
    
    return f_score * (1 - penalty)


In [16]:
def translate_text(text):
    inputs = tokenizer(prefix + text, return_tensors="tf").input_ids
    outputs = model.generate(inputs,max_new_tokens=64)
    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text

In [17]:
text = 'a group of singers are singing in the GAP shop'

In [18]:
print(translate_text(text))

I0000 00:00:1731782353.102012      97 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Aik tabqa gaane walon ka GAP shop mein ga rahe hain.


In [19]:
text = 'Sachin tendulkar is a cricket player'

In [20]:
print(translate_text(text))

Sachin tendulkar cricket khiladi hain


In [ ]:
from tqdm import tqdm

test_text_pairs = create_text_pairs(test_df)
references = [pair[1] for pair in test_text_pairs]

hypotheses = [translate_text(pair[0]) for pair in tqdm(test_text_pairs, desc="Translating")]

print("\nEvaluating BLEU Score:")
for ref, hyp in zip(references, hypotheses):
    print(f"Reference: {ref}")
    print(f"Hypothesis: {hyp}")
    print()  

# Calculate BLEU and METEOR scores
bleu = sacrebleu.corpus_bleu(hypotheses, [references])
mean_bleu = bleu.score

print("\nCalculating METEOR Scores:")
meteor_scores = []
for ref, hyp in zip(references, hypotheses):
    meteor = custom_meteor_score(ref, hyp)
    meteor_scores.append(meteor)
    print(f"Reference: {ref}")
    print(f"Hypothesis: {hyp}")
    print(f"METEOR Score: {meteor}")
    print()  

mean_meteor = sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0

# Print evaluation results
print(f"\nFinal Evaluation Results:")
print(f"Mean BLEU Score: {mean_bleu}")
print(f"Mean METEOR Score: {mean_meteor}")


Translating: 100%|██████████| 3757/3757 [7:37:32<00:00,  7.31s/it]



Evaluating BLEU Score:
Reference: Usne apna safar naye shehron se lekar shaandar pahaadon tak ka ittifaqan tajawuz karte hue tanha raston ka manzarnama tay kiya
Hypothesis: Usne apne tawajju se tay kiya, shor sharaba wale sheher ki manzar se le kar sukoon bhari pahaadon ke kinare tak.

Reference: Ek shaam, jab gas lamps ne dhundhla hawa mein chamakna shuru kiya, Sullivan ne ek raazila paighaam paaya.
Hypothesis: Aik shaam, jab gas lamps smog se bhari hawa mein chamak rahe thay, Sullivan ko aik raazmand paigham milti thi.

Reference: Suraag rasaan ne aik sahasi irada ke saath tehqiqat shuru ki.
Hypothesis: Ek lohe bhari iraada ke saath, jasoos ne sachai ko khulaasa karne ka irada kiya.

Reference: Sanati manzar ne Sullivan ke insaaf ki talash mein apni jagah bana li.
Hypothesis: Sanati manzar Sullivan ke insaaf ki talash mein ek khushk manzar faraham karti thi.

Reference: jasoos aur sheher ke rehaishi ke darmiyan guftagu ne anokhi technologies jo khadab hokar be zabtigyon ki jhadab ki

In [22]:
# model.save_pretrained("/kaggle/working/model")
# tokenizer.save_pretrained("/kaggle/working/tokenizer")  # Save the tokenizer too for consistency
